In [ ]:
# Importar las libreria necesarias que vamos a utilizar
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

In [2]:
# 1. Crear la conexión a una base de datos SQLite en memoria (o en un archivo local)
engine = create_engine('sqlite:///techventas.db')

In [3]:
# 2. Leer el archivo CSV usando Pandas
df = pd.read_csv("../data/ventas_techventas.csv", encoding='utf-8')
df.head()


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
2,"1003,2024-01-10,C003,DataSolutions,Centro,Moni...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz


In [5]:
# 3. Guardar el DataFrame en la base de datos como una tabla llamada 'ventas'
df.to_sql('ventas', con=engine, if_exists='replace', index=False)
print("¡Base de datos y tabla 'ventas' creadas con éxito!")

¡Base de datos y tabla 'ventas' creadas con éxito!


In [6]:
# Creamos la tabla vendedores para el ejercico Q5
with engine.connect() as conn:
    # Crear la tabla utilizando text()
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS vendedores (
        vendedor TEXT PRIMARY KEY,
        zona TEXT,
        meta_mensual REAL
    );
    """))
    
    # Limpiar datos previos si existen utilizando text()
    conn.execute(text("DELETE FROM vendedores;"))
    
    # Insertar los nuevos registros utilizando text()
    conn.execute(text("""
    INSERT INTO vendedores (vendedor, zona, meta_mensual) VALUES
    ('Ana López', 'Norte', 8000),
    ('Carlos Ruiz', 'Sur', 7500),
    ('Luis Mora', 'Norte', 8000),
    ('Maria Torres', 'Centro', 7000);
    """))
    
    # En SQLAlchemy 2.0 es buena práctica confirmar los cambios (commit) 
    # si la conexión no está en modo autocommit.
    conn.commit() 
    
    print("Tabla 'vendedores' creada e insertada correctamente.")

Tabla 'vendedores' creada e insertada correctamente.


### Q1

In [7]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT DISTINCT producto AS PRODCUTOS_DISPONIBLES
        FROM ventas;
    """, 
    con=conn)

print(df_resultado)



  PRODCUTOS_DISPONIBLES
0         Laptop Pro 15
1     Mouse Inalambrico
2                   NaN
3      Teclado Mecanico
4               SSD 1TB
5         Router WiFi 6


### Q2

In [13]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT cliente_id, fecha, cantidad
        FROM ventas
        WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31' AND cantidad > 5;
    """, 
    con=conn)
    
print(df_resultado)


   cliente_id       fecha  cantidad
0        C002  2024-01-07      15.0
1        C001  2024-01-12      10.0
2        C005  2024-01-18      20.0
3        C006  2024-01-22       8.0
4        C003  2024-01-25      12.0
5        C008  2024-02-05      30.0
6        C005  2024-02-12      25.0
7        C002  2024-02-20      18.0
8        C007  2024-02-22      12.0
9        C001  2024-03-04      30.0
10       C009  2024-03-07       8.0
11       C005  2024-03-15      20.0
12       C006  2024-03-18      15.0
13       C007  2024-03-20       6.0
14       C008  2024-03-25      10.0


### Q3

In [14]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT vendedor, SUM(cantidad * precio_unitario) AS ingreso_bruto_total
        FROM ventas
        GROUP BY vendedor
        HAVING ingreso_bruto_total > 10000;
    """, 
    con=conn)

print(df_resultado)

       vendedor  ingreso_bruto_total
0    Ana Lï¿½ï¿              20810.0
1   Carlos Ruiz              21355.0
2     Luis Mora              19830.0
3  Maria Torres              11860.0


### Q4

In [15]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT categoria, COUNT(*) AS TOTAL_PEDIDOS, SUM(cantidad) AS TOTAL_UNIDADES_VENDIDAS, AVG(precio_unitario) AS PROMEDIO_UNITARIO
        FROM ventas
        GROUP BY categoria;
    """, 
    con=conn)

print(df_resultado)

        categoria  TOTAL_PEDIDOS  TOTAL_UNIDADES_VENDIDAS  PROMEDIO_UNITARIO
0             NaN             11                      NaN                NaN
1  Almacenamiento             12                    260.0          95.000000
2     Electronica             14                     31.0        1139.285714
3     Perifericos             15                    215.0          53.000000
4           Redes              8                     39.0         120.000000


### Q5

In [16]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT v.vendedor,
               v.zona,
               v.meta_mensual,
               SUM(f.cantidad * f.precio_unitario) AS ventas_totales,
               (SUM(f.cantidad * f.precio_unitario) / v.meta_mensual) * 100.0 AS porcentaje_cumplimiento
        FROM vendedores v
        INNER JOIN ventas f ON v.vendedor = f.vendedor
        GROUP BY v.vendedor, v.zona, v.meta_mensual;
    """, 
    con=conn)

print(df_resultado)

       vendedor    zona  meta_mensual  ventas_totales  porcentaje_cumplimiento
0   Carlos Ruiz     Sur        7500.0         21355.0               284.733333
1     Luis Mora   Norte        8000.0         19830.0               247.875000
2  Maria Torres  Centro        7000.0         11860.0               169.428571


### Q6

In [17]:
with engine.connect() as conn:
    df_resultado = pd.read_sql("""
        SELECT cliente_id, cliente_nombre, SUM(cantidad * precio_unitario) AS INGRESO_TOTAL
        FROM ventas
        WHERE fecha BETWEEN '2024-01-01' AND '2024-06-31'
        GROUP BY cliente_id
        ORDER BY INGRESO_TOTAL DESC
        LIMIT 1;
    """, 
    con=conn)

print(df_resultado)


  cliente_id cliente_nombre  INGRESO_TOTAL
0       C001    TechCorp SA        19025.0
